In [ ]:
file_path = "dataset/aerofoil_128/2032c.dat.npy"  # change to your file
img = np.load(file_path)

x_resolution = 128
y_resolution = 128
x_lower_lim = -0.01
x_upper_lim =  1.01
y_lower_lim = -0.17
y_upper_lim =  0.27
edge_threshold = 0.002 
xs = np.linspace(x_lower_lim, x_upper_lim, x_resolution)
ys = np.linspace(y_lower_lim, y_upper_lim, y_resolution)
X, Y = np.meshgrid(xs, ys)


plt.figure(figsize=(5,5))
plt.imshow(img, origin="lower", cmap="gray", extent=[x_lower_lim, x_upper_lim, y_lower_lim, y_upper_lim])
plt.title(file_path)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from shapely.geometry import Point, Polygon
from tqdm import tqdm
import re
import os

# ==========================================================
# CONFIGURATION
# ==========================================================
x_resolution = 128
y_resolution = 128
x_lower_lim = -0.01
x_upper_lim =  1.01
y_lower_lim = -0.17
y_upper_lim =  0.27
edge_threshold = 0.002   # distance tolerance to mark pixel as edge
save_dir = "dataset/aerofoil_128"
os.makedirs(save_dir, exist_ok=True)

df = pd.read_csv("dataset/aerofoil_data_clean.csv")

# ==========================================================
# HELPERS
# ==========================================================
def parse_array(s):
    """Convert string like '[0.1 0.2 0.3]' to np array."""
    s = re.sub(r"[\[\]]", "", str(s))
    return np.array([float(v) for v in s.split()], dtype=float)

unique_names = df['name'].unique()
print(f"Detected {len(unique_names)} unique airfoil shapes")

# Setup raster grid
xs = np.linspace(x_lower_lim, x_upper_lim, x_resolution)
ys = np.linspace(y_lower_lim, y_upper_lim, y_resolution)
X, Y = np.meshgrid(xs, ys)

# ==========================================================
# PASS — generate binary edge map
# ==========================================================
for name in tqdm(unique_names, desc="Generate grayscale edge images"):
    sub = df[df['name'] == name].iloc[0]

    # polygon coordinates
    x_vals = parse_array(sub['x'])
    y_vals = parse_array(sub['y'])
    coords = [(x_vals[i], y_vals[i]) for i in range(len(x_vals))]
    poly = Polygon(coords)

    img = np.zeros_like(X, dtype=np.float32)

    # check each pixel
    for i in range(y_resolution):
        for j in range(x_resolution):
            p = Point(X[i, j], Y[i, j])
            
            # distance to border
            dist = p.distance(poly.exterior)

            # mark edge -> 1
            if dist < edge_threshold:
                img[i, j] = 1.0
            else:
                img[i, j] = 0.0

    np.save(f"{save_dir}/{name}.npy", img)

print("DONE: All binary edge matrices saved.")